# Bot Server — Fast Detection, Persisted Calibration

This is the bot-side notebook. Runs on each JetBot. 

## 🚦 Quick start

1. Set `my_own_color` in §2.0 to this bot's cylinder color.
2. Run §1 → §3 (imports, parameters, calibration loader). Calibration loads from disk if present.
3. Run §4 (detection function definitions).
4. Run §5 (smoke test). Confirm cube + peers detect cleanly and check the reported `detection_loop_hz`.
5. If first-time setup: run §6.1 (cube calibration) and §6.2 (peer calibration). Each saves to disk.
6. If lighting changed: run §7 HSV recalibration helpers. Each saves to disk.
7. Run §8 → §9 → §10 to start the detection thread and HTTP server.
8. Run §11.1 to verify detection rate via the `/health` endpoint.
9. Run §12 to shut down cleanly.


## §1. Imports and hardware initialization

In [ ]:
import cv2
import numpy as np
import time
import threading
import math
import json as json_module
import os
from http.server import BaseHTTPRequestHandler, HTTPServer
import urllib.parse
from IPython.display import Image as IPyImage, display, clear_output

from jetbot import Camera, Robot

# Full sensor capture (per lecturer's advice — gives crisp colours, room for cropping).
CAMERA_FULL_WIDTH_PIXELS  = 3000
CAMERA_FULL_HEIGHT_PIXELS = 2000

camera_instance = Camera.instance(
    width=CAMERA_FULL_WIDTH_PIXELS, height=CAMERA_FULL_HEIGHT_PIXELS)
robot_instance = Robot()

print('Camera and robot initialized.')
print(f'  Full frame shape: {camera_instance.value.shape}  (height, width, 3)')


## §2. Tunable parameters

### §2.0 ⚠️ EDIT THIS PER-BOT ⚠️

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  SET THIS TO THIS BOT'S OWN CYLINDER COLOR — EDIT PER BOT!       ║
# ╚══════════════════════════════════════════════════════════════════╝
my_own_color = 'yellow'   # one of: 'purple', 'blue', 'yellow', 'orange'

valid_peer_colors = {'purple', 'blue', 'yellow', 'orange'}
assert my_own_color in valid_peer_colors, (
    f'my_own_color must be one of {valid_peer_colors}, got {my_own_color!r}')
print(f'This bot wears the {my_own_color.upper()} cylinder. '
      f'It will skip {my_own_color} when scanning peers.')


### §2.1 HSV color ranges (defaults — overwritten by saved calibration if present)

In [ ]:
# These are the defaults. If /tmp/jetbot_calibration.json exists, §3 overwrites
# them with the saved values. Run §7 to recalibrate one color at a time.

color_hsv_ranges = {
    'red': [
        (np.array([  0, 100,  60], dtype=np.uint8), np.array([ 10, 255, 255], dtype=np.uint8)),
        (np.array([170, 100,  60], dtype=np.uint8), np.array([180, 255, 255], dtype=np.uint8)),
    ],
    'orange': [
        # Tight max-saturation cap keeps deep-red cube pixels OUT of the orange mask.
        (np.array([ 13,  80,  80], dtype=np.uint8), np.array([ 22, 220, 255], dtype=np.uint8)),
    ],
    'yellow': [
        (np.array([ 25,  80,  80], dtype=np.uint8), np.array([ 38, 255, 255], dtype=np.uint8)),
    ],
    'blue': [
        (np.array([ 88,  80,  60], dtype=np.uint8), np.array([115, 255, 255], dtype=np.uint8)),
    ],
    'purple': [
        (np.array([125,  50,  60], dtype=np.uint8), np.array([155, 255, 255], dtype=np.uint8)),
    ],
}

# Display BGR colors used for drawing bounding boxes in /camera_annotated.
display_bgr_for_color = {
    'red':    (  0,   0, 255),
    'orange': (  0, 140, 255),
    'yellow': (  0, 255, 255),
    'blue':   (255, 100,   0),
    'purple': (200,   0, 200),
}

print('HSV defaults loaded for:', list(color_hsv_ranges.keys()))


### §2.2 Shape filters per object type

All min-area / aspect / solidity thresholds operate on the **downsampled detection frame** (see §2.5). They are sized accordingly: at 600×220 detection resolution, a 5 cm cube at 2 m fills ~25×25 ≈ 600 px.


In [ ]:
# Cube: square-ish.
cube_minimum_blob_area_pixels   = 200   # downsampled detection-frame pixels
cube_minimum_aspect_ratio       = 0.35
cube_maximum_aspect_ratio       = 2.80
cube_minimum_solidity           = 0.55

# Peer cylinder: tall, narrow. bbox aspect = width / height; a 13×30 cm cylinder ≈ 0.43.
peer_minimum_blob_area_pixels   = 120
peer_minimum_aspect_ratio       = 0.18
peer_maximum_aspect_ratio       = 0.80
peer_minimum_solidity           = 0.55

# Scoring weights — same for cube and peer.
scoring_weight_for_area      = 0.5
scoring_weight_for_solidity  = 0.3
scoring_weight_for_squareness = 0.2

# Morphology kernels (in downsampled frame pixels).
morphology_open_kernel_size  = 3
morphology_close_kernel_size = 5

print('Shape filters loaded (downsampled detection-frame coords).')


### §2.3 Distance calibration constants — in CROPPED frame coordinates

These constants are tied to the **cropped frame** (3000×1100), not the downsampled detection frame. The detection function back-scales measurements before applying these constants, so the user-facing calibration is independent of the downsample factor. You can change `detection_target_width_pixels` in §2.5 without re-running §6.


In [ ]:
# Cube: distance_meters = cube_distance_calibration_constant / sqrt(area_in_cropped_pixels)
# Starting guess for 3000×1100 cropped. Run §6.1 to set this from actual hardware.
cube_distance_calibration_constant = 130.0

# Peer cylinder: distance_meters = chart_distance_calibration_constant / bbox_height_in_cropped_pixels
chart_distance_calibration_constant = 600.0

print(f'k_cube  (default) = {cube_distance_calibration_constant}')
print(f'k_chart (default) = {chart_distance_calibration_constant}')
print('  (§3 will overwrite from /tmp/jetbot_calibration.json if it exists)')


### §2.4 Loop period, network, watchdog

In [ ]:
# How often the detection thread refreshes latest_sensor_state.
# Target 10 Hz. Actual rate is reported in /health and printed by §11.1.
detection_loop_period_seconds = 0.07   # asking for ~14 Hz; actual will be lower

# HTTP server.
http_server_port = 8080

# Command watchdog: stops motors if no fresh /command in this many ms.
# Loosened slightly from 500 to 600 to tolerate occasional control-loop hiccups
# without the motors twitching.
default_command_ttl_milliseconds = 600

print('Network and timing parameters loaded.')


### §2.5 Frame crop + detection downsample

`crop_*_pixels` removes ceiling/floor from the captured frame. `detection_target_width_pixels` sets the size of the downsampled frame that detection actually runs on. The downsample is purely a speed optimization — calibration constants live in cropped-frame coordinates and are scaled by the detection function.

Tune `crop_top_pixels` higher to drop more ceiling; lower to see further forward.

Tune `detection_target_width_pixels` higher for accuracy, lower for speed. 600 is a good balance.


In [ ]:
crop_top_pixels    = 700
crop_bottom_pixels = 200
crop_left_pixels   = 0
crop_right_pixels  = 0

assert crop_top_pixels + crop_bottom_pixels < CAMERA_FULL_HEIGHT_PIXELS, (
    f'Crop too aggressive: top + bottom >= full height {CAMERA_FULL_HEIGHT_PIXELS}')

CROPPED_WIDTH_PIXELS  = CAMERA_FULL_WIDTH_PIXELS  - crop_left_pixels - crop_right_pixels
CROPPED_HEIGHT_PIXELS = CAMERA_FULL_HEIGHT_PIXELS - crop_top_pixels  - crop_bottom_pixels

# Detection-frame size — downsample of cropped frame for speed.
detection_target_width_pixels = 600
# Compute the scale factor so the downsampled frame keeps the cropped aspect ratio.
DETECTION_WIDTH_PIXELS  = detection_target_width_pixels
DETECTION_HEIGHT_PIXELS = int(round(
    CROPPED_HEIGHT_PIXELS * (detection_target_width_pixels / CROPPED_WIDTH_PIXELS)))
# Scale from detection-frame pixels back to cropped-frame pixels.
# 3000/600 = 5.0 — a height of 1 px in detection ≈ 5 px in cropped.
detection_to_cropped_scale = CROPPED_WIDTH_PIXELS / DETECTION_WIDTH_PIXELS


def apply_frame_crop(full_frame_bgr):
    '''Crop ceiling + floor off the full sensor frame.'''
    h = full_frame_bgr.shape[0]
    w = full_frame_bgr.shape[1]
    return full_frame_bgr[
        crop_top_pixels  : h - crop_bottom_pixels,
        crop_left_pixels : w - crop_right_pixels,
    ]


def downsample_for_detection(cropped_frame_bgr):
    '''Reduce the cropped frame to detection size for fast HSV/contour work.
    INTER_AREA is the right choice when shrinking.'''
    return cv2.resize(
        cropped_frame_bgr,
        (DETECTION_WIDTH_PIXELS, DETECTION_HEIGHT_PIXELS),
        interpolation=cv2.INTER_AREA)


print('Crop + downsample configured:')
print(f'  Full frame:      {CAMERA_FULL_WIDTH_PIXELS} × {CAMERA_FULL_HEIGHT_PIXELS}')
print(f'  Cropped frame:   {CROPPED_WIDTH_PIXELS} × {CROPPED_HEIGHT_PIXELS}  '
      f'({CROPPED_WIDTH_PIXELS * CROPPED_HEIGHT_PIXELS / 1e6:.1f} MP)')
print(f'  Detection frame: {DETECTION_WIDTH_PIXELS} × {DETECTION_HEIGHT_PIXELS}  '
      f'(downsample factor {detection_to_cropped_scale:.2f}×)')


### §2.6 Detection sanity filters

Distance estimates above these thresholds are treated as false positives (faraway wall pixels matching the red mask, etc.) and dropped. Tune for your room.


In [ ]:
cube_maximum_valid_distance_meters = 5.0
peer_maximum_valid_distance_meters = 3.0

# Below these CROPPED-frame measurements the distance estimate is unreliable
# and the detection is probably background noise.
peer_minimum_bbox_height_in_cropped_pixels = 80
cube_minimum_bbox_area_in_cropped_pixels   = 1200

print('Sanity filters loaded.')
print(f'  Cube: distance < {cube_maximum_valid_distance_meters} m '
      f'AND cropped-area >= {cube_minimum_bbox_area_in_cropped_pixels} px')
print(f'  Peer: distance < {peer_maximum_valid_distance_meters} m '
      f'AND cropped-height >= {peer_minimum_bbox_height_in_cropped_pixels} px')


## §3. Calibration persistence

Calibration values (k_cube, k_chart, HSV ranges) are saved to `/tmp/jetbot_calibration.json` after every `calibrate_*` and `recalibrate_*` call. Running this cell on notebook startup loads the saved values if the file exists, so re-running the notebook doesn't lose hours of calibration.

`/tmp` is writable on JetBots and survives across notebook restarts within the same boot. After a JetBot reboot, the file is gone — you'd recalibrate. (Move the file under `~` if you want it to survive reboots, but `/tmp` is fine for a session.)


In [ ]:
CALIBRATION_FILE_PATH = '/tmp/jetbot_calibration.json'


def save_calibration_to_disk():
    '''Persist k_cube, k_chart, and HSV ranges to disk.'''
    payload = {
        'cube_distance_calibration_constant':  cube_distance_calibration_constant,
        'chart_distance_calibration_constant': chart_distance_calibration_constant,
        # numpy arrays aren't JSON serializable directly; convert to lists.
        'color_hsv_ranges': {
            color: [(lo.tolist(), hi.tolist()) for (lo, hi) in ranges]
            for color, ranges in color_hsv_ranges.items()
        },
    }
    try:
        with open(CALIBRATION_FILE_PATH, 'w') as f:
            json_module.dump(payload, f, indent=2)
        print(f'Calibration saved to {CALIBRATION_FILE_PATH}.')
    except OSError as save_error:
        print(f'WARNING: could not save calibration: {save_error!r}')


def load_calibration_from_disk():
    '''Overwrite k_cube, k_chart, and HSV ranges from disk if file exists.'''
    global cube_distance_calibration_constant, chart_distance_calibration_constant
    if not os.path.exists(CALIBRATION_FILE_PATH):
        print(f'No saved calibration at {CALIBRATION_FILE_PATH}. Using defaults from §2.')
        return False
    try:
        with open(CALIBRATION_FILE_PATH, 'r') as f:
            payload = json_module.load(f)
    except (OSError, ValueError) as load_error:
        print(f'WARNING: could not load calibration: {load_error!r} — using defaults.')
        return False
    cube_distance_calibration_constant  = float(payload.get(
        'cube_distance_calibration_constant', cube_distance_calibration_constant))
    chart_distance_calibration_constant = float(payload.get(
        'chart_distance_calibration_constant', chart_distance_calibration_constant))
    saved_hsv = payload.get('color_hsv_ranges', {})
    for color, ranges in saved_hsv.items():
        if color in color_hsv_ranges:
            color_hsv_ranges[color] = [
                (np.array(lo, dtype=np.uint8), np.array(hi, dtype=np.uint8))
                for (lo, hi) in ranges]
    print(f'Calibration LOADED from {CALIBRATION_FILE_PATH}:')
    print(f'  k_cube  = {cube_distance_calibration_constant}')
    print(f'  k_chart = {chart_distance_calibration_constant}')
    print(f'  HSV ranges restored for: {list(saved_hsv.keys())}')
    return True


# Run the load now.
load_calibration_from_disk()


## §4. Detection function

**Key change vs v2:** the HSV conversion happens ONCE per frame, then every color reuses that HSV image. v2 was doing 5 separate `cvtColor` calls on the full cropped frame — that was the main reason detection ran at 3.5 Hz.

Coordinates returned in `/state` are normalized `[-1, +1]` on the detection frame, which is the same as `[-1, +1]` on the cropped frame (normalization is resolution-invariant). Bounding boxes returned for the cube and peers are in **cropped-frame** coordinates, suitable for direct drawing on the cropped frame in §9 annotation.


In [ ]:
def build_mask_for_color_from_hsv(frame_in_hsv, color_name):
    '''Build a binary mask for a single colour from the SHARED HSV frame.
    The input is already-converted HSV (computed once per frame).'''
    accumulated_mask = None
    for lower_hsv, upper_hsv in color_hsv_ranges[color_name]:
        range_mask = cv2.inRange(frame_in_hsv, lower_hsv, upper_hsv)
        if accumulated_mask is None:
            accumulated_mask = range_mask
        else:
            accumulated_mask = cv2.bitwise_or(accumulated_mask, range_mask)

    open_kernel  = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (morphology_open_kernel_size, morphology_open_kernel_size))
    close_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (morphology_close_kernel_size, morphology_close_kernel_size))
    cleaned_mask = cv2.morphologyEx(accumulated_mask, cv2.MORPH_OPEN,  open_kernel)
    cleaned_mask = cv2.morphologyEx(cleaned_mask,    cv2.MORPH_CLOSE, close_kernel)
    return cleaned_mask


def score_candidate_contour(candidate_contour, object_type):
    '''Returns (score, properties) where score is None if the candidate was filtered out.
    Operates in detection-frame coordinates throughout.'''
    if object_type == 'cube':
        min_area = cube_minimum_blob_area_pixels
        min_ar   = cube_minimum_aspect_ratio
        max_ar   = cube_maximum_aspect_ratio
        min_sol  = cube_minimum_solidity
        expected_aspect_ratio = 1.0
    else:  # peer
        min_area = peer_minimum_blob_area_pixels
        min_ar   = peer_minimum_aspect_ratio
        max_ar   = peer_maximum_aspect_ratio
        min_sol  = peer_minimum_solidity
        expected_aspect_ratio = 0.43

    candidate_area_pixels = float(cv2.contourArea(candidate_contour))
    bx, by, bw, bh = cv2.boundingRect(candidate_contour)
    aspect_ratio = bw / max(bh, 1)

    convex_hull = cv2.convexHull(candidate_contour)
    convex_hull_area = float(cv2.contourArea(convex_hull))
    solidity = candidate_area_pixels / max(convex_hull_area, 1.0)

    properties = {
        'area':         candidate_area_pixels,    # detection-frame
        'aspect_ratio': aspect_ratio,
        'solidity':     solidity,
        'bounding_box': (bx, by, bw, bh),         # detection-frame
        'rejection_reason': None,
    }

    if candidate_area_pixels < min_area:
        properties['rejection_reason'] = f'area {candidate_area_pixels:.0f} < {min_area}'
        return None, properties
    if not (min_ar <= aspect_ratio <= max_ar):
        properties['rejection_reason'] = f'aspect {aspect_ratio:.2f} outside [{min_ar},{max_ar}]'
        return None, properties
    if solidity < min_sol:
        properties['rejection_reason'] = f'solidity {solidity:.2f} < {min_sol}'
        return None, properties

    detection_frame_area_pixels = DETECTION_WIDTH_PIXELS * DETECTION_HEIGHT_PIXELS
    normalized_area_score = min(candidate_area_pixels / (detection_frame_area_pixels * 0.25), 1.0)
    shape_score = 1.0 - min(
        abs(math.log(aspect_ratio / expected_aspect_ratio)) / math.log(2.8), 1.0)

    final_score = (
        scoring_weight_for_area      * normalized_area_score +
        scoring_weight_for_solidity  * solidity +
        scoring_weight_for_squareness * shape_score
    )
    properties['score'] = final_score
    return final_score, properties


def find_best_contour_for_color(frame_in_hsv, color_name, object_type):
    '''Build mask, find contours, score them, return best + a debug record.'''
    color_mask = build_mask_for_color_from_hsv(frame_in_hsv, color_name)
    contours, _ = cv2.findContours(color_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best_score = -1.0
    best_properties = None
    candidates_seen = []
    for candidate_contour in contours:
        score, properties = score_candidate_contour(candidate_contour, object_type)
        debug_record = {k: v for k, v in properties.items() if k != 'contour'}
        debug_record['accepted'] = score is not None
        candidates_seen.append(debug_record)
        if score is not None and score > best_score:
            best_score = score
            best_properties = properties
            best_properties['contour'] = candidate_contour

    return best_properties, {
        'total_contours_found': len(contours),
        'candidates_evaluated': candidates_seen[:10],
    }


def detect_cube_and_peers(detection_frame_bgr):
    '''Main detection entry point. Input: the DOWNSAMPLED cropped frame.

    Returns {'cube': dict_or_None, 'peers': list_of_dicts, '_debug': dict}.
    All bounding boxes are returned in CROPPED-frame coordinates (upscaled here)
    so the annotator can draw on the cropped frame directly.
    Distance constants are in CROPPED-frame units, so the measurements are
    back-scaled before applying them.
    '''
    debug_log = {'red_debug': None, 'peer_debug': {}, 'rejected_by_sanity': []}

    # ONE HSV conversion for the whole frame, shared by all colours.
    frame_in_hsv = cv2.cvtColor(detection_frame_bgr, cv2.COLOR_BGR2HSV)

    # ── Cube ───────────────────────────────────────────────────────────
    cube_properties, red_debug = find_best_contour_for_color(frame_in_hsv, 'red', 'cube')
    debug_log['red_debug'] = red_debug

    cube_result = None
    if cube_properties is not None:
        contour_moments = cv2.moments(cube_properties['contour'])
        if contour_moments['m00'] > 0:
            centroid_x = contour_moments['m10'] / contour_moments['m00']
            centroid_y = contour_moments['m01'] / contour_moments['m00']
            cube_x_norm = (centroid_x - DETECTION_WIDTH_PIXELS  / 2) / (DETECTION_WIDTH_PIXELS  / 2)
            cube_y_norm = (DETECTION_HEIGHT_PIXELS / 2 - centroid_y) / (DETECTION_HEIGHT_PIXELS / 2)

            # Back-scale measurements into cropped-frame coords for the calibration
            # constant to apply correctly.
            area_in_cropped = cube_properties['area'] * (detection_to_cropped_scale ** 2)
            # cube_distance = cube_distance_calibration_constant / math.sqrt(area_in_cropped) SOS
            raw_distance_estimate = cube_distance_calibration_constant / math.sqrt(area_in_cropped)
            # Linear correction from three-point calibration (physical vs reported).
            # Fit: physical = 2.4385 * reported - 0.0714. Residuals ≤ 3.3 mm across 0.3–0.7m.
            cube_distance = max(0.0, 2.4385 * raw_distance_estimate - 0.0714)

            # Upscale bbox to cropped-frame coords for annotation.
            bx, by, bw, bh = cube_properties['bounding_box']
            cropped_bbox = tuple(int(round(v * detection_to_cropped_scale))
                                  for v in (bx, by, bw, bh))

            passes_distance = cube_distance < cube_maximum_valid_distance_meters
            passes_area     = area_in_cropped >= cube_minimum_bbox_area_in_cropped_pixels

            if passes_distance and passes_area:
                cube_result = {
                    'cube_x_normalized':     cube_x_norm,
                    'cube_y_normalized':     cube_y_norm,
                    'cube_blob_area_pixels': area_in_cropped,       # cropped-frame
                    'cube_distance_meters':  cube_distance,
                    'cube_bounding_box':     cropped_bbox,          # cropped-frame
                    'cube_solidity':         cube_properties['solidity'],
                    'cube_score':            cube_properties['score'],
                }
            else:
                reasons = []
                if not passes_distance:
                    reasons.append(f'distance {cube_distance:.2f} m > {cube_maximum_valid_distance_meters}')
                if not passes_area:
                    reasons.append(f'cropped-area {area_in_cropped:.0f} < {cube_minimum_bbox_area_in_cropped_pixels}')
                debug_log['rejected_by_sanity'].append({
                    'object': 'cube',
                    'distance_estimate': cube_distance,
                    'cropped_area':      area_in_cropped,
                    'reason':            '; '.join(reasons),
                })

    # ── Peers ──────────────────────────────────────────────────────────
    peer_results = []
    for peer_color in valid_peer_colors:
        if peer_color == my_own_color:
            continue
        peer_properties, peer_debug = find_best_contour_for_color(frame_in_hsv, peer_color, 'peer')
        debug_log['peer_debug'][peer_color] = peer_debug
        if peer_properties is None:
            continue

        bx, by, bw, bh = peer_properties['bounding_box']
        peer_x_norm = (bx + bw / 2.0 - DETECTION_WIDTH_PIXELS / 2) / (DETECTION_WIDTH_PIXELS / 2)

        # Back-scale measurements into cropped-frame coords.
        bbox_height_in_cropped = bh * detection_to_cropped_scale
        peer_distance = chart_distance_calibration_constant / max(bbox_height_in_cropped, 1.0)

        cropped_bbox = tuple(int(round(v * detection_to_cropped_scale))
                              for v in (bx, by, bw, bh))

        passes_distance = peer_distance < peer_maximum_valid_distance_meters
        passes_height   = bbox_height_in_cropped >= peer_minimum_bbox_height_in_cropped_pixels

        if passes_distance and passes_height:
            peer_results.append({
                'peer_color':                       peer_color,
                'peer_x_normalized':                peer_x_norm,
                'peer_distance_meters':             peer_distance,
                'peer_bounding_box':                cropped_bbox,
                'peer_bbox_height_pixels':          bbox_height_in_cropped,  # cropped-frame
                'peer_solidity':                    peer_properties['solidity'],
                'peer_score':                       peer_properties['score'],
            })
        else:
            reasons = []
            if not passes_distance:
                reasons.append(f'distance {peer_distance:.2f} m > {peer_maximum_valid_distance_meters}')
            if not passes_height:
                reasons.append(f'cropped-height {bbox_height_in_cropped:.0f} < {peer_minimum_bbox_height_in_cropped_pixels}')
            debug_log['rejected_by_sanity'].append({
                'object':            f'peer_{peer_color}',
                'distance_estimate': peer_distance,
                'cropped_height':    bbox_height_in_cropped,
                'reason':            '; '.join(reasons),
            })

    return {'cube': cube_result, 'peers': peer_results, '_debug': debug_log}


print('Detection functions defined (single cvtColor, downsampled detection, '
      'cropped-frame bboxes returned).')


## §5. Smoke test + detection-rate benchmark

Runs the full capture→crop→downsample→detect pipeline 30 times, prints what was detected and the achieved rate. Displays the most recent annotated cropped frame inline.

Target: >= 10 Hz. If you see <7 Hz, the bot is CPU-bound — drop `detection_target_width_pixels` in §2.5 to 480, restart §2.5 → §4 → §5.


In [ ]:
def annotate_cropped_frame_with_detections(cropped_frame_bgr, detection_result):
    '''Draw cube + peer bounding boxes on the cropped frame.
    Bboxes in detection_result are already in cropped-frame coords.'''
    annotated = cropped_frame_bgr.copy()

    cube = detection_result.get('cube')
    if cube is not None:
        bx, by, bw, bh = cube['cube_bounding_box']
        cv2.rectangle(annotated, (bx, by), (bx + bw, by + bh),
                      display_bgr_for_color['red'], 8)
        label_text = (f'CUBE {cube["cube_distance_meters"]:.2f}m '
                      f'x={cube["cube_x_normalized"]:+.2f}')
        cv2.putText(annotated, label_text, (bx, max(40, by - 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.4,
                    display_bgr_for_color['red'], 3, cv2.LINE_AA)

    for peer in detection_result.get('peers', []):
        bx, by, bw, bh = peer['peer_bounding_box']
        color_bgr = display_bgr_for_color[peer['peer_color']]
        cv2.rectangle(annotated, (bx, by), (bx + bw, by + bh), color_bgr, 6)
        label_text = (f'{peer["peer_color"].upper()} '
                      f'{peer["peer_distance_meters"]:.2f}m')
        cv2.putText(annotated, label_text, (bx, max(40, by - 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, color_bgr, 3, cv2.LINE_AA)

    return annotated


def bgr8_to_jpeg(frame_bgr, quality=80):
    return cv2.imencode('.jpg', frame_bgr, [cv2.IMWRITE_JPEG_QUALITY, quality])[1].tobytes()


def smoke_test_detection(samples=30):
    '''Run the full pipeline N times, report rate, display last annotated frame.'''
    last_annotated = None
    last_detection = None
    samples_with_cube  = 0
    samples_with_peers = 0

    started = time.time()
    for _ in range(samples):
        full = camera_instance.value
        cropped = apply_frame_crop(full)
        detection_frame = downsample_for_detection(cropped)
        detection = detect_cube_and_peers(detection_frame)
        if detection['cube'] is not None:
            samples_with_cube += 1
        if detection['peers']:
            samples_with_peers += 1
        last_annotated = annotate_cropped_frame_with_detections(cropped, detection)
        last_detection = detection
    elapsed = time.time() - started

    measured_hz = samples / elapsed
    print(f'Smoke test: {samples} iterations in {elapsed:.2f} s '
          f'→ {measured_hz:.1f} Hz')
    print(f'  Cube seen in  {samples_with_cube}/{samples} samples')
    print(f'  Peers seen in {samples_with_peers}/{samples} samples')
    if last_detection:
        if last_detection['cube'] is not None:
            c = last_detection['cube']
            print(f'  Last cube: distance={c["cube_distance_meters"]:.2f} m, '
                  f'x={c["cube_x_normalized"]:+.2f}, area={c["cube_blob_area_pixels"]:.0f}')
        if last_detection['peers']:
            for p in last_detection['peers']:
                print(f'  Last peer {p["peer_color"]:>6}: '
                      f'distance={p["peer_distance_meters"]:.2f} m, '
                      f'x={p["peer_x_normalized"]:+.2f}, '
                      f'h={p["peer_bbox_height_pixels"]:.0f}')

    if last_annotated is not None:
        # Resize for inline display so we don't blow up the notebook.
        display_h = 360
        h, w = last_annotated.shape[:2]
        display_w = int(w * display_h / h)
        smaller = cv2.resize(last_annotated, (display_w, display_h),
                             interpolation=cv2.INTER_AREA)
        display(IPyImage(data=bgr8_to_jpeg(smaller, 75)))


# Run smoke test now.
smoke_test_detection(samples=30)


### §5.1 Live preview (optional)

Streams the annotated cropped frame inline for ~15 s. Useful for moving the cube around and verifying detection across positions. Stop early by interrupting the kernel.


In [ ]:
def run_live_preview(duration_seconds=15.0):
    started = time.time()
    while time.time() - started < duration_seconds:
        full = camera_instance.value
        cropped = apply_frame_crop(full)
        detection_frame = downsample_for_detection(cropped)
        detection = detect_cube_and_peers(detection_frame)
        annotated = annotate_cropped_frame_with_detections(cropped, detection)
        display_h = 360
        h, w = annotated.shape[:2]
        smaller = cv2.resize(
            annotated, (int(w * display_h / h), display_h), interpolation=cv2.INTER_AREA)
        clear_output(wait=True)
        display(IPyImage(data=bgr8_to_jpeg(smaller, 75)))


# Uncomment to use:
# run_live_preview(duration_seconds=15.0)
print('run_live_preview() ready — uncomment the call above to use it.')


## §6. Distance calibration

Each calibration helper averages N detection samples, back-solves the calibration constant, and **persists to disk** automatically. You can re-run the notebook tomorrow and skip these unless lighting or geometry changed.


### §6.1 Cube distance calibration

Place the red cube at a precisely measured distance in front of the bot. Then run the helper with that distance in meters.


In [ ]:
def calibrate_cube_distance_constant(known_distance_meters, samples_to_average=15):
    '''Place the cube at `known_distance_meters` first, then run this.'''
    global cube_distance_calibration_constant
    areas_in_cropped_frame = []
    for sample_index in range(samples_to_average):
        full = camera_instance.value
        cropped = apply_frame_crop(full)
        detection_frame = downsample_for_detection(cropped)
        detection = detect_cube_and_peers(detection_frame)
        if detection['cube'] is not None:
            areas_in_cropped_frame.append(detection['cube']['cube_blob_area_pixels'])
        time.sleep(0.05)
    if not areas_in_cropped_frame:
        print('FAILED — cube was not detected in any sample. Check HSV ranges first (§7).')
        return None
    mean_area = float(np.mean(areas_in_cropped_frame))
    new_k = known_distance_meters * math.sqrt(mean_area)
    cube_distance_calibration_constant = new_k
    print(f'Mean cube area (cropped-frame): {mean_area:.0f} px from '
          f'{len(areas_in_cropped_frame)}/{samples_to_average} samples')
    print(f'  → cube_distance_calibration_constant = {new_k:.2f}')
    save_calibration_to_disk()
    return new_k


# UNCOMMENT, set the actual measured distance, then run:
# calibrate_cube_distance_constant(known_distance_meters=1.00, samples_to_average=15)
print('calibrate_cube_distance_constant() ready.')


### §6.2 Peer (cylinder) distance calibration

Place ONE peer cylinder at a precisely measured distance in front of the bot. Then run with that distance.


In [ ]:
def calibrate_chart_distance_constant(known_distance_meters, samples_to_average=15):
    '''Place ONE peer cylinder at known_distance_meters first.'''
    global chart_distance_calibration_constant
    heights_in_cropped_frame = []
    for sample_index in range(samples_to_average):
        full = camera_instance.value
        cropped = apply_frame_crop(full)
        detection_frame = downsample_for_detection(cropped)
        detection = detect_cube_and_peers(detection_frame)
        for peer in detection['peers']:
            heights_in_cropped_frame.append(peer['peer_bbox_height_pixels'])
        time.sleep(0.05)
    if not heights_in_cropped_frame:
        print('FAILED — no peer cylinder detected. Verify §7 HSV ranges for the peer color first.')
        return None
    mean_height = float(np.mean(heights_in_cropped_frame))
    new_k = known_distance_meters * mean_height
    chart_distance_calibration_constant = new_k
    print(f'Mean peer bbox height (cropped-frame): {mean_height:.0f} px from '
          f'{len(heights_in_cropped_frame)} samples')
    print(f'  → chart_distance_calibration_constant = {new_k:.2f}')
    save_calibration_to_disk()
    return new_k


# UNCOMMENT, set the actual measured distance, then run:
# calibrate_chart_distance_constant(known_distance_meters=1.00, samples_to_average=15)
print('calibrate_chart_distance_constant() ready.')


## §7. Per-color HSV recalibration

Auto-fit HSV bounds for one color from samples of that color in front of the camera. Useful when lighting changes between rooms.


In [ ]:
def recalibrate_color_hsv(color_name, samples_to_collect=20,
                          saturation_floor=80, value_floor=60,
                          hue_margin=4, sv_margin=20):
    '''Place ONE object of `color_name` in front of the camera, then run.
    Auto-fits HSV bounds from the largest contour in each sample.
    For red, wraparound is handled (returns two ranges).'''
    global color_hsv_ranges
    hue_samples = []
    sat_samples = []
    val_samples = []
    for _ in range(samples_to_collect):
        full = camera_instance.value
        cropped = apply_frame_crop(full)
        detection_frame = downsample_for_detection(cropped)
        frame_in_hsv = cv2.cvtColor(detection_frame, cv2.COLOR_BGR2HSV)

        # Permissive seed mask: just the original ranges for this color.
        seed_mask = build_mask_for_color_from_hsv(frame_in_hsv, color_name)
        contours, _ = cv2.findContours(seed_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            time.sleep(0.05)
            continue
        biggest = max(contours, key=cv2.contourArea)
        single_blob_mask = np.zeros_like(seed_mask)
        cv2.drawContours(single_blob_mask, [biggest], -1, 255, -1)
        pixels = frame_in_hsv[single_blob_mask > 0]
        if pixels.size == 0:
            time.sleep(0.05)
            continue
        hue_samples.extend(pixels[:, 0].tolist())
        sat_samples.extend(pixels[:, 1].tolist())
        val_samples.extend(pixels[:, 2].tolist())
        time.sleep(0.05)

    if not hue_samples:
        print(f'FAILED — no {color_name} pixels found. Check that the object is in frame.')
        return None

    hues = np.array(hue_samples)
    sats = np.array(sat_samples)
    vals = np.array(val_samples)

    sat_low = max(saturation_floor, int(np.percentile(sats, 5))  - sv_margin)
    sat_low = max(0, sat_low)
    sat_high = 255
    val_low = max(value_floor,    int(np.percentile(vals, 5))  - sv_margin)
    val_low = max(0, val_low)
    val_high = 255

    if color_name == 'red':
        # Split hue at 90 — red wraps around 0/180.
        lo_band = hues[hues < 90]
        hi_band = hues[hues >= 90]
        new_ranges = []
        if lo_band.size:
            h_lo = max(0,   int(np.percentile(lo_band, 5))  - hue_margin)
            h_hi = min(179, int(np.percentile(lo_band, 95)) + hue_margin)
            new_ranges.append((np.array([h_lo, sat_low, val_low], dtype=np.uint8),
                                np.array([h_hi, sat_high, val_high], dtype=np.uint8)))
        if hi_band.size:
            h_lo = max(0,   int(np.percentile(hi_band, 5))  - hue_margin)
            h_hi = min(179, int(np.percentile(hi_band, 95)) + hue_margin)
            new_ranges.append((np.array([h_lo, sat_low, val_low], dtype=np.uint8),
                                np.array([h_hi, sat_high, val_high], dtype=np.uint8)))
    else:
        h_lo = max(0,   int(np.percentile(hues, 5))  - hue_margin)
        h_hi = min(179, int(np.percentile(hues, 95)) + hue_margin)
        new_ranges = [(np.array([h_lo, sat_low, val_low], dtype=np.uint8),
                       np.array([h_hi, sat_high, val_high], dtype=np.uint8))]

    color_hsv_ranges[color_name] = new_ranges
    print(f'New HSV ranges for {color_name}:')
    for lo, hi in new_ranges:
        print(f'  H {lo[0]:3d}-{hi[0]:3d}   S {lo[1]:3d}-{hi[1]:3d}   V {lo[2]:3d}-{hi[2]:3d}')
    save_calibration_to_disk()
    return new_ranges


# Uncomment to recalibrate one color at a time:
# recalibrate_color_hsv('red')
# recalibrate_color_hsv('blue')
# recalibrate_color_hsv('yellow')
# recalibrate_color_hsv('purple')
# recalibrate_color_hsv('orange')
print('recalibrate_color_hsv() ready.')


## §8. Background detection thread

Captures, crops, downsamples, detects, computes frame difference, updates shared state. Runs at `detection_loop_period_seconds` (or as fast as the pipeline allows).


In [ ]:
sensor_state_lock = threading.Lock()

latest_detection_result   = {'cube': None, 'peers': []}
latest_cropped_frame_bgr  = None
latest_detection_debug    = {}

latest_sensor_state = {
    'timestamp_seconds':         0.0,
    'bot_own_color':             my_own_color,
    'cube_visible':              False,
    'cube_x_normalized':         0.0,
    'cube_y_normalized':         0.0,
    'cube_distance_meters':     -1.0,
    'frame_difference_value':    0.0,
    'peers':                     [],
    'detection_loop_iteration':  0,
    'detection_loop_hz':         0.0,
}

detection_thread_should_run = False


def compute_frame_difference(current_frame, previous_frame):
    '''Mean absolute pixel difference, downsampled to 32x32 for speed.'''
    if previous_frame is None:
        return 0.0
    small_current  = cv2.resize(current_frame,  (32, 32))
    small_previous = cv2.resize(previous_frame, (32, 32))
    return float(np.mean(cv2.absdiff(small_current, small_previous)))


def run_detection_loop():
    global latest_sensor_state, latest_detection_result
    global latest_cropped_frame_bgr, latest_detection_debug

    previous_detection_frame = None
    iteration_counter = 0
    rate_window_started = time.time()
    rate_window_frames  = 0
    latest_measured_hz  = 0.0

    while detection_thread_should_run:
        iteration_started = time.time()

        # 1. Capture full sensor frame.
        captured_full_frame = camera_instance.value
        # 2. Crop ceiling + floor.
        captured_cropped_frame = apply_frame_crop(captured_full_frame)
        # 3. Downsample for detection.
        detection_frame = downsample_for_detection(captured_cropped_frame)
        # 4. Run detection.
        detection_result = detect_cube_and_peers(detection_frame)
        # 5. Frame difference for stuck-detection (computed on detection frame for speed).
        frame_difference = compute_frame_difference(detection_frame, previous_detection_frame)
        previous_detection_frame = detection_frame.copy()

        iteration_counter += 1
        rate_window_frames += 1
        if time.time() - rate_window_started >= 1.0:
            latest_measured_hz = rate_window_frames / (time.time() - rate_window_started)
            rate_window_started = time.time()
            rate_window_frames  = 0

        if detection_result['cube'] is None:
            cube_state = {
                'cube_visible':         False,
                'cube_x_normalized':    0.0,
                'cube_y_normalized':    0.0,
                'cube_distance_meters': -1.0,
            }
        else:
            cube_state = {
                'cube_visible':         True,
                'cube_x_normalized':    detection_result['cube']['cube_x_normalized'],
                'cube_y_normalized':    detection_result['cube']['cube_y_normalized'],
                'cube_distance_meters': detection_result['cube']['cube_distance_meters'],
                'cube_bounding_box':    [int(v) for v in detection_result['cube']['cube_bounding_box']],
            }

        peers_for_state = [
            {
                'peer_color':              p['peer_color'],
                'peer_x_normalized':       p['peer_x_normalized'],
                'peer_distance_meters':    p['peer_distance_meters'],
                'peer_bbox_height_pixels': int(p['peer_bbox_height_pixels']),
            }
            for p in detection_result['peers']
        ]

        new_state = {
            'timestamp_seconds':         time.time(),
            'bot_own_color':             my_own_color,
            **cube_state,
            'frame_difference_value':    frame_difference,
            'peers':                     peers_for_state,
            'detection_loop_iteration':  iteration_counter,
            'detection_loop_hz':         latest_measured_hz,
        }

        with sensor_state_lock:
            latest_sensor_state       = new_state
            latest_detection_result   = {
                'cube':  detection_result['cube'],
                'peers': detection_result['peers'],
            }
            latest_cropped_frame_bgr  = captured_cropped_frame
            latest_detection_debug    = detection_result.get('_debug', {})

        time_elapsed = time.time() - iteration_started
        sleep_seconds = detection_loop_period_seconds - time_elapsed
        if sleep_seconds > 0:
            time.sleep(sleep_seconds)


def start_detection_thread():
    global detection_thread_should_run
    if detection_thread_should_run:
        print('Detection thread already running.')
        return
    detection_thread_should_run = True
    threading.Thread(target=run_detection_loop, daemon=True).start()
    print('Detection thread started.')


def stop_detection_thread():
    global detection_thread_should_run
    detection_thread_should_run = False
    print('Detection thread stop requested.')


print('Detection thread helpers defined.')


## §9. HTTP server

Endpoints (all unchanged from v2 in shape, but `/camera_annotated` now draws on the high-res cropped frame for crisp debug-view visuals):

- `GET  /state`              — JSON sensor snapshot
- `GET  /camera`             — full uncropped frame (debugging the crop bounds)
- `GET  /camera_annotated`   — cropped frame with cube + peer bboxes drawn
- `GET  /health`             — detection thread status + rate
- `GET  /detection_debug`    — last rejection reasons for diagnostics
- `POST /command`            — `{"left_motor_speed": float, "right_motor_speed": float, "command_ttl_milliseconds": int}`


In [ ]:
command_watchdog_lock = threading.Lock()
active_command_watchdog_timer = None


def apply_motor_command(left_motor_speed, right_motor_speed, command_ttl_milliseconds):
    '''Set motors, then arm a watchdog that stops them after TTL if no fresh command arrives.'''
    global active_command_watchdog_timer
    robot_instance.set_motors(float(left_motor_speed), float(right_motor_speed))
    with command_watchdog_lock:
        if active_command_watchdog_timer is not None:
            active_command_watchdog_timer.cancel()
        watchdog_seconds = max(0.05, command_ttl_milliseconds / 1000.0)
        new_timer = threading.Timer(watchdog_seconds, lambda: robot_instance.stop())
        new_timer.daemon = True
        new_timer.start()
        active_command_watchdog_timer = new_timer


def cancel_command_watchdog_and_stop():
    global active_command_watchdog_timer
    with command_watchdog_lock:
        if active_command_watchdog_timer is not None:
            active_command_watchdog_timer.cancel()
            active_command_watchdog_timer = None
    robot_instance.stop()


STREAM_DISPLAY_WIDTH_PIXELS = 900


def resize_for_streaming(frame_bgr):
    '''Shrink a frame to a reasonable width for HTTP streaming.'''
    h, w = frame_bgr.shape[:2]
    if w <= STREAM_DISPLAY_WIDTH_PIXELS:
        return frame_bgr
    new_h = int(h * STREAM_DISPLAY_WIDTH_PIXELS / w)
    return cv2.resize(frame_bgr, (STREAM_DISPLAY_WIDTH_PIXELS, new_h),
                       interpolation=cv2.INTER_AREA)


class JetbotSensorRequestHandler(BaseHTTPRequestHandler):
    def log_message(self, format_string, *args):
        return  # quiet

    def _respond_json(self, status_code, payload):
        body = json_module.dumps(payload).encode('utf-8')
        self.send_response(status_code)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Content-Length', str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def _respond_text(self, status_code, message):
        body = message.encode('utf-8')
        self.send_response(status_code)
        self.send_header('Content-Type', 'text/plain')
        self.send_header('Content-Length', str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def _respond_jpeg(self, jpeg_bytes):
        self.send_response(200)
        self.send_header('Content-Type', 'image/jpeg')
        self.send_header('Content-Length', str(len(jpeg_bytes)))
        self.end_headers()
        self.wfile.write(jpeg_bytes)

    def do_GET(self):
        path = urllib.parse.urlparse(self.path).path

        if path == '/state':
            with sensor_state_lock:
                snapshot = dict(latest_sensor_state)
            self._respond_json(200, snapshot)

        elif path == '/camera':
            full = camera_instance.value
            self._respond_jpeg(bgr8_to_jpeg(resize_for_streaming(full), 70))

        elif path == '/camera_annotated':
            with sensor_state_lock:
                cached_cropped = (latest_cropped_frame_bgr.copy()
                                  if latest_cropped_frame_bgr is not None else None)
                detection_snapshot = {
                    'cube':  latest_detection_result.get('cube'),
                    'peers': list(latest_detection_result.get('peers', [])),
                }
            if cached_cropped is None:
                cached_cropped = apply_frame_crop(camera_instance.value)
            annotated = annotate_cropped_frame_with_detections(cached_cropped, detection_snapshot)
            self._respond_jpeg(bgr8_to_jpeg(resize_for_streaming(annotated), 70))

        elif path == '/health':
            with sensor_state_lock:
                iteration = latest_sensor_state.get('detection_loop_iteration', 0)
                hz        = latest_sensor_state.get('detection_loop_hz', 0.0)
            if hz >= 8.0:    status_label = 'good'
            elif hz >= 5.0:  status_label = 'marginal'
            elif hz > 0:     status_label = 'too_slow'
            else:            status_label = 'no_data_yet'
            self._respond_json(200, {
                'status':                    'ok',
                'bot_own_color':             my_own_color,
                'detection_thread_running':  detection_thread_should_run,
                'detection_loop_iteration':  iteration,
                'detection_loop_hz':         hz,
                'detection_rate_status':     status_label,
                'cropped_frame_width':       CROPPED_WIDTH_PIXELS,
                'cropped_frame_height':      CROPPED_HEIGHT_PIXELS,
                'detection_frame_width':     DETECTION_WIDTH_PIXELS,
                'detection_frame_height':    DETECTION_HEIGHT_PIXELS,
            })

        elif path == '/detection_debug':
            with sensor_state_lock:
                debug_snapshot = dict(latest_detection_debug)
            self._respond_json(200, debug_snapshot)

        else:
            self._respond_text(404, f'Unknown path: {path}')

    def do_POST(self):
        path = urllib.parse.urlparse(self.path).path
        if path == '/command':
            try:
                body_length = int(self.headers.get('Content-Length', '0'))
                raw         = self.rfile.read(body_length)
                parsed_body = json_module.loads(raw.decode('utf-8'))
                left_motor  = float(parsed_body['left_motor_speed'])
                right_motor = float(parsed_body['right_motor_speed'])
                ttl_ms      = int(parsed_body.get('command_ttl_milliseconds',
                                                  default_command_ttl_milliseconds))
                left_motor  = max(-1.0, min(1.0, left_motor))
                right_motor = max(-1.0, min(1.0, right_motor))
                apply_motor_command(left_motor, right_motor, ttl_ms)
                self._respond_json(200, {
                    'accepted_left_motor_speed':  left_motor,
                    'accepted_right_motor_speed': right_motor,
                    'watchdog_ttl_milliseconds':  ttl_ms,
                })
            except (ValueError, KeyError, json_module.JSONDecodeError) as parse_error:
                self._respond_json(400, {'error': f'bad command body: {parse_error!r}'})
        else:
            self._respond_text(404, f'Unknown path: {path}')


http_server_instance = None
http_server_thread   = None


def start_http_server():
    global http_server_instance, http_server_thread
    if http_server_instance is not None:
        print('HTTP server already running.')
        return
    http_server_instance = HTTPServer(('0.0.0.0', http_server_port), JetbotSensorRequestHandler)
    http_server_thread   = threading.Thread(
        target=http_server_instance.serve_forever, daemon=True)
    http_server_thread.start()
    print(f'HTTP server listening on port {http_server_port}.')


def stop_http_server():
    global http_server_instance, http_server_thread
    if http_server_instance is None:
        print('HTTP server not running.')
        return
    http_server_instance.shutdown()
    http_server_instance.server_close()
    http_server_instance = None
    http_server_thread   = None
    print('HTTP server stopped.')


print('HTTP server helpers defined.')


## §10. Start the server

In [ ]:
start_detection_thread()
time.sleep(0.4)   # give the thread a moment to capture+detect once.
start_http_server()
print(f'\nBot {my_own_color.upper()} ready. Endpoints:')
print(f'  http://<this-bot-ip>:{http_server_port}/state')
print(f'  http://<this-bot-ip>:{http_server_port}/camera_annotated')
print(f'  http://<this-bot-ip>:{http_server_port}/health')


### §11.1 Detection-rate check

Reads `/health` and prints the measured Hz. Should be >= 8 for "good" status. If you see "too_slow", consider lowering `detection_target_width_pixels` in §2.5 (then re-run §2.5, §4, §5, §8, §9, §10).


In [ ]:
import urllib.request

try:
    with urllib.request.urlopen(f'http://127.0.0.1:{http_server_port}/health',
                                 timeout=2.0) as response:
        health_payload = json_module.loads(response.read().decode('utf-8'))
    print(f'Detection rate     : {health_payload["detection_loop_hz"]:.1f} Hz '
          f'({health_payload["detection_rate_status"]})')
    print(f'Detection iteration: {health_payload["detection_loop_iteration"]}')
    print(f'Cropped frame      : '
          f'{health_payload["cropped_frame_width"]}×{health_payload["cropped_frame_height"]}')
    print(f'Detection frame    : '
          f'{health_payload["detection_frame_width"]}×{health_payload["detection_frame_height"]}')
except Exception as health_error:
    print(f'Could not reach /health locally: {health_error!r}')


## §12. Shutdown

In [ ]:
cancel_command_watchdog_and_stop()
stop_http_server()
stop_detection_thread()
print('Bot shut down cleanly.')
